# Open-XTTS-Trainer: three-stage demo

A short walk through the recipe on a tiny sample set (6 LJSpeech clips plus JFK, about 1 minute of speech).
It shows the steps and the commands, not results: losses on 1 minute of data say nothing about quality.
Results come from the evaluation harness, an A/B comparison against stock XTTS-v2 (`scripts/compare_ab.py`;
docs/recipe_design.md §6, phase 8).

What the recipe trains, and when (docs/recipe_design.md §3):
1. **Stage 2, GPT:** the core of any fine-tune: voice adaptation, or a new language with an extended vocab.
2. **Stage 3, HiFi-GAN decoder:** optional; keep it only if it beats the stock decoder in the A/B.
3. **Stage 1, DVAE:** off by default. A trained DVAE changes what the GPT's audio tokens mean, so it forces a
   full Stage 2 retrain.

Licenses: XTTS-v2 weights are under the Coqui Public Model License (non-commercial). The forced aligner
(MMS_FA) and ECAPA2 are CC BY-NC 4.0.

## 1. Install (pinned) and fetch the base model (hash-verified)

In [ ]:
import os, json
!nvidia-smi
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .   # pins coqui-tts to the commit in pyproject.toml
!python scripts/fetch_base_artifacts.py --dest xtts_base   # fails on any hash or size mismatch

## 2. Data: transcribe, align, cut at pauses
Every clip's text is exactly the words spoken in it, and the command fails unless the result passes validation.

In [ ]:
!python -m xtts_trainer.speech_data --source sample --output_dir sample_audio
# Your own recording instead (language and speaker are required):
# !python -m xtts_trainer.speech_data --input_file my_voice.wav --lang en --speaker_name me --output_dir sample_audio
print(json.dumps(json.load(open('sample_audio/prep_report.json'))['recordings'], indent=1)[:2000])

## 3. Stage 2: GPT fine-tune through upstream GPTTrainer
The best checkpoint by eval loss is exported to `checkpoints/gpt/export/`, a complete checkpoint directory.
For a new language: extend the vocab (`xtts_trainer/stage2_gpt/vocab_extender.py --languages xx`), then pass
`--vocab_path` and `--languages xx`, plus `--replay` with base-language data.

In [ ]:
!python -m xtts_trainer.stage2_gpt.train --dataset sample_audio/dataset.jsonl --base_dir xtts_base \
    --output_dir checkpoints/gpt --epochs 2 --batch_size 2 --grad_accum_steps 1 > stage2.log 2>&1 || tail -40 stage2.log
print(json.load(open('checkpoints/gpt/export/training_summary.json')))

## 4. Stage 3 (optional): decoder fine-tune on latents from the Stage 2 model
Latents must come from the GPT you will run, so they are extracted from the Stage 2 export.

In [ ]:
!python -m xtts_trainer.stage3_hifigan.extract_latents --dataset sample_audio/dataset.jsonl \
    --checkpoint_dir checkpoints/gpt/export --output_dir latents
!python -m xtts_trainer.stage3_hifigan.train --dataset sample_audio/dataset.jsonl --latents_dir latents \
    --checkpoint_dir checkpoints/gpt/export --output_dir checkpoints/hifigan \
    --epochs 60 --batch_size 4 --disc_warmup_steps 40
!tail -3 checkpoints/hifigan/metrics.jsonl

## 5. Stage 1 (off by default): DVAE diagnostic first
The diagnostic compares the stock DVAE on your data against reference speech. Fine-tune only if it fits your
audio clearly worse, and then retrain Stage 2 on the new codes (`--retrain_for_new_dvae`).

In [ ]:
!python scripts/diagnose_dvae.py --dataset sample_audio/dataset.jsonl --checkpoint xtts_base/dvae.pth
# !python -m xtts_trainer.stage1_dvae.train --dataset sample_audio/dataset.jsonl --base_checkpoint xtts_base/dvae.pth \
#     --output_dir checkpoints/dvae --epochs 20 --batch_size 4

## 6. Listen: Stage 2 model, with the stock decoder and with the Stage 3 decoder

In [ ]:
import torch, soundfile as sf, IPython.display as ipd
from xtts_trainer.models import load_state_strict, safe_torch_load
from xtts_trainer.stage2_gpt.export import load_for_inference

model = load_for_inference('checkpoints/gpt/export', 'cuda')
reference = sorted(r['audio'] for r in map(json.loads, open('sample_audio/dataset.jsonl').readlines()[1:]))[0]
text = 'Artificial intelligence is rapidly transforming how we communicate every day.'
lat, spk = model.get_conditioning_latents(audio_path=[f'sample_audio/{reference}'])
for name in ('stock_decoder', 'stage3_decoder'):
    if name == 'stage3_decoder':
        load_state_strict(model.hifigan_decoder, safe_torch_load('checkpoints/hifigan/hifigan.pth')['hifigan_decoder'], 'Stage 3 decoder')
    torch.manual_seed(42)
    wav = model.inference(text, 'en', lat, spk, temperature=0.7)['wav']
    sf.write(f'{name}.wav', wav, 24000)
    print(name); display(ipd.Audio(f'{name}.wav'))

## 7. Measure (Phase 8)
Measure the fine-tuned model against stock XTTS-v2 on the frozen eval set, with paired statistics:
```
python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --languages en --output eval/baselines/base.json
python scripts/run_eval_baseline.py --checkpoint_dir checkpoints/gpt/export --languages en --output eval/baselines/tuned.json
python scripts/compare_ab.py eval/baselines/base.json eval/baselines/tuned.json --gates eval/gates.json
```